In [21]:
import os
import re
import pickle as pkl
import ast
from collections import Counter
from typing import List, Tuple, Any
from tqdm import tqdm
import time

import pandas as pd
import numpy as np


import matplotlib.pyplot as plt

from llm_unsupervised_conf.plots import METHODS_MAP, METHODS_MAP_SHORT, DATASETS_MAP

plt.style.use('seaborn-v0_8')
pal = plt.rcParams['axes.prop_cycle'].by_key()['color']

In [22]:
shifts = ["qa", "math", "language"]

full_df = []
for shift in shifts:
    df = pd.read_csv(f"../results/{shift}_shift.csv")
    full_df.append(df)
full_df = pd.concat(full_df)
# full_df["Method"] = [METHODS_MAP[m] for m in full_df["Method"].tolist()]
full_df

,Shift,Method,ECE1,ECE2,MCE,Brier,AUROC
0,QA Domain,ans_logprob,0.373477,0.386080,0.539644,0.384307,0.571989
1,QA Domain,logprob,0.256894,0.282622,0.442060,0.300673,0.666915
2,QA Domain,oracle_sc,0.095045,0.113165,0.209996,0.197284,0.749739
3,QA Domain,ridge_clip,0.139832,0.168034,0.309854,0.245765,0.656382
4,QA Domain,split_isotonic_on_ridge,0.115519,0.134381,0.244847,0.234701,0.655346
5,QA Domain,split_isotonic_on_ridge_nrt,0.117958,0.133773,0.259985,0.231870,0.658630
6,QA Domain,verbal_conf,0.333601,0.352598,0.545500,0.345416,0.663258
0,Math Domain,ans_logprob,0.132959,0.171414,0.379085,0.132963,0.673517
1,Math Domain,logprob,0.103327,0.113454,0.172474,0.121445,0.640059
2,Math Domain,oracle_sc,0.048166,0.069043,0.169998,0.075664,0.832819


In [23]:
full_df[["Shift", "Method", "ECE2", "Brier"]]

,Shift,Method,ECE2,Brier
0,QA Domain,ans_logprob,0.386080,0.384307
1,QA Domain,logprob,0.282622,0.300673
2,QA Domain,oracle_sc,0.113165,0.197284
3,QA Domain,ridge_clip,0.168034,0.245765
4,QA Domain,split_isotonic_on_ridge,0.134381,0.234701
5,QA Domain,split_isotonic_on_ridge_nrt,0.133773,0.231870
6,QA Domain,verbal_conf,0.352598,0.345416
0,Math Domain,ans_logprob,0.171414,0.132963
1,Math Domain,logprob,0.113454,0.121445
2,Math Domain,oracle_sc,0.069043,0.075664


In [24]:
OUR_METHOD = "split_isotonic_on_ridge_nrt"
comp_methods = ["logprob", "ans_logprob", "verbal_conf", OUR_METHOD]
tile_df = full_df[full_df["Method"].isin(comp_methods)]

tiled = (
    tile_df[["Shift", "Method", "ECE2", "Brier"]]
    .set_index(["Method", "Shift"])
    .unstack("Shift")                      # columns become Metric -> Shift
    .swaplevel(0, 1, axis=1)              # columns become Shift -> Metric
    .sort_index(axis=1, level=0)          # sort shifts
    .reindex(["ECE2", "Brier"], axis=1, level=1)  # metric order within each shift
    .reindex(comp_methods, axis=0)   # sort rows by comp_methods
)
tiled = tiled.rename(index=METHODS_MAP)
tiled

Shift         Language           Math Domain           QA Domain          
                  ECE2     Brier        ECE2     Brier      ECE2     Brier
Method                                                                    
Token Probs.  0.152613  0.165237    0.113454  0.121445  0.282622  0.300673
Ans. Probs.   0.238294  0.189858    0.171414  0.132963  0.386080  0.384307
Verbal Conf.  0.182403  0.180817    0.150570  0.137678  0.352598  0.345416
Ours          0.092268  0.123494    0.095348  0.110677  0.133773  0.231870

In [26]:
print(tiled.reset_index().to_latex(index=False, float_format="%.3f"))

\begin{tabular}{lrrrrrr}
\toprule
Method & \multicolumn{2}{r}{Language} & \multicolumn{2}{r}{Math Domain} & \multicolumn{2}{r}{QA Domain} \\
 & ECE2 & Brier & ECE2 & Brier & ECE2 & Brier \\
\midrule
Token Probs. & 0.153 & 0.165 & 0.113 & 0.121 & 0.283 & 0.301 \\
Ans. Probs. & 0.238 & 0.190 & 0.171 & 0.133 & 0.386 & 0.384 \\
Verbal Conf. & 0.182 & 0.181 & 0.151 & 0.138 & 0.353 & 0.345 \\
Ours & 0.092 & 0.123 & 0.095 & 0.111 & 0.134 & 0.232 \\
\bottomrule
\end{tabular}

